In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import librosa

NOTE_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'E#', 'G', 'B#', 'A', 'A#', 'B']

# Khoảng cách (semitone) từ nốt gốc cho từng loại hợp âm
CHORD_TYPES = {
    '':     [0, 4, 7],
    'm':    [0, 3, 7],
    'maj7': [0, 4, 7, 11],
    '7':    [0, 4, 7, 10],
    'm7':   [0, 3, 7, 10],
}

def _build_templates():
    t = {}
    for root in range(12):
        for suffix, intervals in CHORD_TYPES.items():
            v = np.zeros(12)
            for i in intervals:
                v[(root + i) % 12] = 1
            t[(root, suffix)] = (v / np.linalg.norm(v), intervals)
    return t

TEMPLATES = _build_templates()


def get_bass_note(y, sr):
    """Nốt thấp nhất có năng lượng đáng kể trong dải C1–B3 (CQT)."""
    C = np.abs(librosa.cqt(y, sr=sr, fmin=librosa.note_to_hz('C1'),
                           n_bins=36, bins_per_octave=12))
    energy = C.mean(axis=1)
    if energy.max() < 1e-6:
        return None
    idx = np.where(energy > 0.4 * energy.max())[0][0]
    return idx % 12  # bin 0 = C1 nên idx % 12 = cao độ lớp nốt

def analyze_single_file(file_path):
    try:
        y, sr = librosa.load(file_path)
        y, _ = librosa.effects.trim(y, top_db=30)

        chroma = librosa.feature.chroma_cqt(y=librosa.effects.harmonic(y), sr=sr)
        energies = chroma.mean(axis=1)
        energies = energies / (energies.max() + 1e-9)   # chuẩn hóa tương đối
        detected = [NOTE_NAMES[i] for i in range(12) if energies[i] > 0.4]

        # Template matching (cosine similarity)
        e = energies / (np.linalg.norm(energies) + 1e-9)
        (root, suffix), (_, intervals) = max(
            TEMPLATES.items(), key=lambda kv: float(e @ kv[1][0]))

        name = NOTE_NAMES[root] + suffix
        bass = get_bass_note(y, sr)
        chord_tones = {(root + i) % 12 for i in intervals}
        if bass is not None and bass != root and bass in chord_tones:
            name += f"/{NOTE_NAMES[bass]}"

        return {"file_name": os.path.basename(file_path),
                "notes": detected,
                "bass": NOTE_NAMES[bass] if bass is not None else None,
                "prediction": name}
    except Exception as ex:
        return {"file_name": os.path.basename(file_path),
                "notes": [], "bass": None, "prediction": f"Lỗi: {ex}"}

def print_table(results, save_csv=None):
    rows = []
    for r in results:
        stem = os.path.splitext(r["file_name"])[0]        # "C.wav" -> "C"
        pred = r["prediction"]
        base_pred = pred.split("/")[0]                    # "C/E" -> "C"
        ok = "✓" if base_pred == stem.split("/")[0] else "✗"
        rows.append({
            "File": r["file_name"],
            "Dự đoán": pred,
            "Bass": r["bass"] or "-",
            "Các nốt": " ".join(r["notes"]),
            "Khớp tên file": ok,
        })

    df = pd.DataFrame(rows)
    print("\n" + "=" * 70)
    print(df.to_string(index=False))
    print("=" * 70)

    total = len(df)
    correct = (df["Khớp tên file"] == "✓").sum()
    print(f"Khớp tên file: {correct}/{total} ({correct / total:.0%})")

    if save_csv:
        df.to_csv(save_csv, index=False, encoding="utf-8-sig")
        print(f"Đã lưu: {save_csv}")
    return df


def process_audio_folder(folder_path, save_csv="ket_qua.csv"):
    files = sorted(glob.glob(os.path.join(folder_path, "*.wav")) +
                   glob.glob(os.path.join(folder_path, "*.mp3")))
    if not files:
        print(f"Không tìm thấy file audio nào trong: {folder_path}")
        return []
    print(f"=== Đang xử lý {len(files)} file... ===")
    results = [analyze_single_file(f) for f in files]
    print_table(results, save_csv=save_csv)
    return results


if __name__ == "__main__":
    FOLDER_PATH = r"D:/machine_learning/final_project/progress/my_dataset/sounds"
    process_audio_folder(FOLDER_PATH)

=== Đang xử lý 35 file... ===

     File  Dự đoán Bass    Các nốt Khớp tên file
    A.wav        A   B#        E A             ✓
   A7.wav       A7   B#        G A             ✓
   Am.wav       Am   B#        E A             ✓
  Am7.wav      Am7   B#        G A             ✓
Amaj7.wav Amaj7/B#   B#       B# A             ✓
    B.wav     B/E#   E#       E# B             ✓
   B7.wav       B7    B  D# E# A B             ✓
   Bm.wav       Bm    B       E# B             ✓
  Bm7.wav      Bm7    B   D E# A B             ✓
Bmaj7.wav    Bmaj7    B D# E# A# B             ✓
    C.wav        C    C      C E G             ✓
   C7.wav       C7    C   C E G A#             ✓
   Cm.wav       Cm    C        C G             ✓
  Cm7.wav      Cm7    C  C D# G A#             ✓
Cmaj7.wav    Cmaj7    C    C E G B             ✓
    D.wav        D    D     D E# A             ✓
   D7.wav       D7    D   C D E# A             ✓
   Dm.wav       Dm    D      D F A             ✓
  Dm7.wav      Dm7    D    C D F A    